# 2026.10.19(월) 20h30~

In [1]:
import os                         # 환경변수 읽기용
from dotenv import load_dotenv    # .env 파일을 읽어 환경변수로 등록하는 함수

load_dotenv()  # .env 파일에서 OPENAI_API_KEY를 읽어 옴

# 키가 없으면 여기서 바로 멈추고 안내 메시지 출력 (뒤에서 알기 어려운 에러가 나는 것을 방지)
assert os.getenv("OPENAI_API_KEY"), ".env 파일에 OPENAI_API_KEY가 없습니다."

# 실습 전체에서 사용할 모델 (gpt-4o-mini는 2027-01-14 종료 예정, 이후에는 여기만 바꾸면 됨)
MODEL = "gpt-4o-mini"

## 1. state 정의

In [2]:
from typing import Optional        # Optional[str]: 값이 없을 수도(None) 있다는 뜻
from pydantic import BaseModel     # 데이터 구조와 타입을 정의하는 클래스

# State: 그래프 안을 흐르는 데이터의 모양
class QAState(BaseModel):
    question: str                    # 사용자 질문 (필수)
    answer: Optional[str] = None     # 생성될 답변 (처음에는 비어 있음)

## 2. node 정의

node는 state를 받아서 바뀐 값만 dictionary로 돌려줌

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, temperature=0)   # temperature=0: 매번 비슷한 답

# Node: State를 입력으로 받아 작업하는 함수
def generate_answer(state: QAState):
    response = llm.invoke(state.question)   # 질문을 모델에 그대로 전달
    return {"answer": response.content}     # 바뀐 값(answer)만 돌려주면 State에 반영됨

## 3. 그래프 구성?

In [4]:
from langgraph.graph import StateGraph, START, END   # 그래프 클래스와 시작/종료 지점

graph = StateGraph(QAState)                          # QAState 모양의 데이터가 흐르는 그래프 생성
graph.add_node("generate_answer", generate_answer)   # ("노드 이름", 실행할 함수) 등록

graph.add_edge(START, "generate_answer")   # 시작점 (graph.set_entry_point(...)와 같음)
graph.add_edge("generate_answer", END)     # 답변 후 종료

app = graph.compile()   # 설계도를 실행 가능한 앱으로 변환

question = "한국의 수도는 어디인가요?"
result = app.invoke({"question": question})   # 초기 State를 넣어 실행 → 최종 State가 딕셔너리로 반환

print("질문:", question)
print("답변:", result["answer"])

질문: 한국의 수도는 어디인가요?
답변: 한국의 수도는 서울입니다.
